# Neural Networks Lab — Truth Seeker Dataset

**Course**: BFOR 516 — Advanced Data Analytics for Cyber  
**Dataset**: Truth_Seeker_Dataset.csv  
**Target**: `BinaryNumTarget` (1 = True claim, 0 = False/Misleading claim)

---

## Overview

We build **two neural network models** to classify whether a fact-checked statement is true:

| Model | Feature Set | Input Dim |
|-------|-------------|-----------|
| Model 1 | TF-IDF (tweet + statement) + Linguistic numerics | ~1,048 |
| Model 2 | User account behavior (followers, bot score, etc.) | 9 |

Both models use **4 hidden layers**, **epochs=50**, **batch_size=64**, and are evaluated with a confusion matrix and accuracy.

In [ ]:
# ── Imports & Reproducibility Seeds ─────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

---
## 1. Load & Inspect Dataset

In [ ]:
# Dataset lives in Week 4/; this notebook is in Week 6/
DATA_PATH = '../Week 4/Truth_Seeker_Dataset.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH}'.\n"
        "Make sure Truth_Seeker_Dataset.csv is in the Week 4 folder."
    )

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(list(df.columns))
print(f"\nTarget distribution:")
print(df['BinaryNumTarget'].value_counts(normalize=True).round(3))
print(f"\nNull counts (columns with nulls only):")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

---
## 2. Preprocessing — Handle Missing Values & Drop Unused Columns

In [ ]:
# Drop columns we won't use in either model
drop_cols = ['embeddings', 'majority_target']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Drop rows where target label is missing
df = df.dropna(subset=['BinaryNumTarget'])

# Fill NaN: empty string for text columns, 0 for all numeric columns
for col in ['tweet', 'statement']:
    if col in df.columns:
        df[col] = df[col].fillna('')

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols] = df[numeric_cols].fillna(0)

print(f"Clean dataset shape: {df.shape}")
print(f"Total remaining nulls: {df.isnull().sum().sum()}")

---
## MODEL 1: Linguistic Features

**Feature set:**
- TF-IDF representation of combined `tweet` + `statement` text (1,000 features)
- 48 numeric linguistic features: tweet engagement stats, NER percentages, POS tag counts, punctuation/word-length stats

**Architecture:** Input(~1,048) → Dense(256) → Dropout(0.3) → Dense(128) → Dropout(0.2) → Dense(64) → Dense(32) → Dense(1, sigmoid)

In [ ]:
# ── Model 1: Feature Engineering ────────────────────────────────────────────

# All numeric linguistic features in the dataset
LINGUISTIC_NUMERIC = [
    # Tweet engagement stats
    'mentions', 'quotes', 'replies', 'retweets', 'favourites', 'hashtags', 'URLs',
    # Token counts
    'unique_count', 'total_count',
    # Named Entity Recognition (NER) percentages
    'ORG_percentage', 'NORP_percentage', 'GPE_percentage', 'PERSON_percentage',
    'MONEY_percentage', 'DATE_percentage', 'CARDINAL_percentage', 'PERCENT_percentage',
    'ORDINAL_percentage', 'FAC_percentage', 'LAW_percentage', 'PRODUCT_percentage',
    'EVENT_percentage', 'TIME_percentage', 'LOC_percentage', 'WORK_OF_ART_percentage',
    'QUANTITY_percentage', 'LANGUAGE_percentage',
    # Word-length stats
    'Word count', 'Max word length', 'Min word length', 'Average word length',
    # POS tag counts
    'present_verbs', 'past_verbs', 'adjectives', 'adverbs', 'adpositions',
    'pronouns', 'TOs', 'determiners', 'conjunctions',
    # Punctuation / character stats
    'dots', 'exclamation', 'questions', 'ampersand', 'capitals', 'digits',
    'long_word_freq', 'short_word_freq'
]
# Keep only columns that actually exist in this version of the dataset
LINGUISTIC_NUMERIC = [c for c in LINGUISTIC_NUMERIC if c in df.columns]

# TF-IDF on combined tweet + statement text
df['combined_text'] = df['tweet'] + ' ' + df['statement']
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['combined_text']).toarray()

# Numeric linguistic features
X_ling_num = df[LINGUISTIC_NUMERIC].values

# Stack TF-IDF + numeric → full feature matrix for Model 1
X_linguistic = np.hstack([X_tfidf, X_ling_num])
y = df['BinaryNumTarget'].values.astype(int)

print(f"TF-IDF features:      {X_tfidf.shape[1]}")
print(f"Numeric linguistic:   {X_ling_num.shape[1]}")
print(f"Total Model 1 inputs: {X_linguistic.shape[1]}")

In [ ]:
# ── Model 1: Train/Test Split + Feature Scaling ──────────────────────────────
X1_train_raw, X1_test_raw, y1_train, y1_test = train_test_split(
    X_linguistic, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # preserve class balance in both splits
)

# StandardScaler normalizes both TF-IDF values and numeric features to the same scale.
# Fit on training data only — transform test data using training statistics.
scaler1 = StandardScaler()
X1_train = scaler1.fit_transform(X1_train_raw)
X1_test  = scaler1.transform(X1_test_raw)

print(f"Train set: {X1_train.shape}")
print(f"Test set:  {X1_test.shape}")
print(f"Train class balance: {np.bincount(y1_train)}  (0s, 1s)")

In [ ]:
# ── Model 1: Architecture ────────────────────────────────────────────────────
# Four hidden layers sized to match high-dimensional TF-IDF + numeric input.
# Dropout after the two largest layers prevents overfitting on rare TF-IDF tokens.

n_inputs_1 = X1_train.shape[1]

model1 = Sequential([
    Input(shape=(n_inputs_1,)),
    Dense(256, activation='relu'),   # Hidden layer 1
    Dropout(0.3),                    # 30% dropout — high-dim input prone to overfit
    Dense(128, activation='relu'),   # Hidden layer 2
    Dropout(0.2),                    # 20% dropout
    Dense(64,  activation='relu'),   # Hidden layer 3
    Dense(32,  activation='relu'),   # Hidden layer 4
    Dense(1,   activation='sigmoid') # Output: probability of True claim
])

model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model1.summary()

In [ ]:
# ── Model 1: Training ────────────────────────────────────────────────────────
history1 = model1.fit(
    X1_train, y1_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,  # 20% of training data used to track overfitting
    verbose=1
)

In [ ]:
# ── Model 1: Evaluation ──────────────────────────────────────────────────────

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history1.history['accuracy'],     label='Train')
axes[0].plot(history1.history['val_accuracy'], label='Validation')
axes[0].set_title('Model 1 — Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history1.history['loss'],     label='Train')
axes[1].plot(history1.history['val_loss'], label='Validation')
axes[1].set_title('Model 1 — Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

# Test set performance
test_loss1, test_acc1 = model1.evaluate(X1_test, y1_test, verbose=0)
y1_pred = (model1.predict(X1_test) >= 0.5).astype(int).flatten()

print(f"\nModel 1 Test Accuracy: {test_acc1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y1_test, y1_pred, target_names=['False (0)', 'True (1)']))

# Confusion matrix
cm1 = confusion_matrix(y1_test, y1_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm1, display_labels=['False (0)', 'True (1)']).plot(
    ax=ax, cmap='Blues'
)
ax.set_title('Model 1 Confusion Matrix — Linguistic Features')
plt.tight_layout()
plt.show()

---
## MODEL 2: User Behavior Features

**Feature set:** 9 numeric account-level signals  
`followers_count`, `friends_count`, `favourites_count`, `statuses_count`, `listed_count`, `following`, `BotScore`, `cred`, `normalize_influence`

**Architecture:** Input(9) → Dense(64) → Dropout(0.2) → Dense(32) → Dense(16) → Dense(8) → Dense(1, sigmoid)

> Architecture is intentionally smaller than Model 1. With only 9 inputs, a 256-wide first layer would create an extreme parameter-to-feature ratio and overfit immediately.

In [ ]:
# ── Model 2: Feature Engineering ────────────────────────────────────────────
USER_BEHAVIOR_COLS = [
    'followers_count',    # Total followers
    'friends_count',      # Accounts the user follows
    'favourites_count',   # Total likes given
    'statuses_count',     # Total tweets posted
    'listed_count',       # Times added to Twitter lists
    'following',          # Binary: does this account follow back
    'BotScore',           # Probability account is a bot [0,1]
    'cred',               # Credibility score
    'normalize_influence' # Normalized influence score
]
# Keep only columns present in this dataset version
USER_BEHAVIOR_COLS = [c for c in USER_BEHAVIOR_COLS if c in df.columns]

X_user = df[USER_BEHAVIOR_COLS].values
y2     = df['BinaryNumTarget'].values.astype(int)

print(f"User behavior features ({len(USER_BEHAVIOR_COLS)}):")
for col in USER_BEHAVIOR_COLS:
    print(f"  {col}")
print(f"\nFeature matrix shape: {X_user.shape}")

In [ ]:
# ── Model 2: Train/Test Split + Feature Scaling ──────────────────────────────
# followers_count can exceed 1M while BotScore is in [0,1] — scaling is critical
X2_train_raw, X2_test_raw, y2_train, y2_test = train_test_split(
    X_user, y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train_raw)
X2_test  = scaler2.transform(X2_test_raw)

print(f"Train set: {X2_train.shape}")
print(f"Test set:  {X2_test.shape}")
print(f"Train class balance: {np.bincount(y2_train)}  (0s, 1s)")

In [ ]:
# ── Model 2: Architecture ────────────────────────────────────────────────────
# Proportionally smaller layers for 9-feature input.
# 64→32→16→8 keeps the parameter count reasonable while still using 4 hidden layers.

n_inputs_2 = X2_train.shape[1]

model2 = Sequential([
    Input(shape=(n_inputs_2,)),
    Dense(64, activation='relu'),    # Hidden layer 1
    Dropout(0.2),                    # Light dropout — structured features, less noise
    Dense(32, activation='relu'),    # Hidden layer 2
    Dense(16, activation='relu'),    # Hidden layer 3
    Dense(8,  activation='relu'),    # Hidden layer 4
    Dense(1,  activation='sigmoid')  # Output: probability of True claim
])

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model2.summary()

In [ ]:
# ── Model 2: Training ────────────────────────────────────────────────────────
history2 = model2.fit(
    X2_train, y2_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# ── Model 2: Evaluation ──────────────────────────────────────────────────────

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history2.history['accuracy'],     label='Train')
axes[0].plot(history2.history['val_accuracy'], label='Validation')
axes[0].set_title('Model 2 — Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history2.history['loss'],     label='Train')
axes[1].plot(history2.history['val_loss'], label='Validation')
axes[1].set_title('Model 2 — Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

# Test set performance
test_loss2, test_acc2 = model2.evaluate(X2_test, y2_test, verbose=0)
y2_pred = (model2.predict(X2_test) >= 0.5).astype(int).flatten()

print(f"\nModel 2 Test Accuracy: {test_acc2:.4f}")
print(f"\nClassification Report:")
print(classification_report(y2_test, y2_pred, target_names=['False (0)', 'True (1)']))

# Confusion matrix
cm2 = confusion_matrix(y2_test, y2_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=['False (0)', 'True (1)']).plot(
    ax=ax, cmap='Oranges'
)
ax.set_title('Model 2 Confusion Matrix — User Behavior Features')
plt.tight_layout()
plt.show()

---
## Model Comparison Summary

In [ ]:
print("=" * 62)
print("  MODEL COMPARISON SUMMARY")
print("=" * 62)
print(f"  {'Model':<35} {'Test Accuracy':>12}")
print("-" * 62)
print(f"  {'Model 1 — Linguistic Features':<35} {test_acc1:>11.4f}")
print(f"  {'Model 2 — User Behavior Features':<35} {test_acc2:>11.4f}")
print("=" * 62)

winner = 'Model 1 (Linguistic)' if test_acc1 >= test_acc2 else 'Model 2 (User Behavior)'
print(f"\nHigher accuracy: {winner}")

---
## Analysis & Report

### Dataset
- **Source**: Truth Seeker Dataset — tweet/statement pairs with fact-check labels
- **Target**: `BinaryNumTarget` (1 = True claim, 0 = False/Misleading claim)
- **Preprocessing**: Dropped `embeddings` and `majority_target` columns; filled NaN with 0 for numerics, empty string for text; dropped rows with missing labels.
- **Split**: 80% training / 20% test, stratified on target class to preserve class balance.

---

### Model 1 — Linguistic Features

**Architecture**: Input(~1,048) → Dense(256, ReLU) → Dropout(0.3) → Dense(128, ReLU) → Dropout(0.2) → Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, Sigmoid)

**Hyperparameter Rationale**:
- **Dense(256) first layer**: TF-IDF creates ~1,000 sparse features. A wide first layer allows the network to compress these into meaningful patterns before progressively narrowing.
- **Dropout(0.3) after layer 1**: High-dimensional TF-IDF is prone to overfitting on rare tokens. 30% dropout forces the network to not rely on any single token.
- **Dropout(0.2) after layer 2**: Lighter regularization once the representation is more compact.
- **No dropout on layers 3 & 4**: At 64 and 32 units, these layers operate on an already-compressed representation — additional dropout would be too aggressive.
- **epochs=50, batch_size=64**: 50 epochs is sufficient for convergence on this dataset size; 64 is a standard batch size offering smooth gradient updates without excessive memory use.
- **validation_split=0.2**: Allows monitoring train vs. validation curves to detect overfitting during training.

---

### Model 2 — User Behavior Features

**Architecture**: Input(9) → Dense(64, ReLU) → Dropout(0.2) → Dense(32, ReLU) → Dense(16, ReLU) → Dense(8, ReLU) → Dense(1, Sigmoid)

**Hyperparameter Rationale**:
- **Dense(64) first layer**: With only 9 input features, starting at 256 would create roughly 2,300 parameters per input feature — a massive overparameterization that would memorize training data. 64 neurons maintains expressive capacity while keeping the ratio reasonable.
- **Tapering 64→32→16→8**: Each layer halves the width, progressively compressing the 9-feature signal into a binary prediction. This is a well-established pattern for small-input networks.
- **Dropout(0.2) only on layer 1**: User behavior features are structured numeric signals (not sparse text), so lighter regularization is appropriate. Dropout on deeper layers would hurt more than it helps given the already-small representation.
- **Same epochs/batch_size as Model 1**: Ensures a fair comparison between the two architectures.

---

### Key Design Decisions (Cross-Model)

| Decision | Rationale |
|----------|----------|
| **StandardScaler on all numeric inputs** | `followers_count` can exceed 1M; `BotScore` is in [0,1]. Without scaling, large-magnitude features dominate gradient updates and cause training instability. |
| **TF-IDF (not raw counts)** | TF-IDF normalizes term frequency by document frequency, reducing the influence of common words that appear across all tweets. |
| **Combined tweet + statement text** | Statement is the fact-check claim; tweet is user commentary — both carry linguistic signal about credibility. |
| **Stratified train/test split** | Ensures both splits reflect the overall class distribution, preventing a biased evaluation. |
| **Random seeds (42)** | Ensures results are reproducible across runs. |
| **sigmoid output + binary_crossentropy** | Standard choice for binary classification — sigmoid maps logit to probability [0,1]; binary_crossentropy measures log-loss against the true binary label. |

---

### Findings & Discussion

*(Fill in after running — replace placeholders with your actual results)*

- **Model 1 (Linguistic)** achieved a test accuracy of **[X.XX]**. The training curves show [describe convergence / overfitting pattern].
- **Model 2 (User Behavior)** achieved a test accuracy of **[X.XX]**. The confusion matrix reveals [describe which class was harder to predict].
- **Which performed better and why**: [Linguistic features likely outperform user behavior features because the content of the statement itself is the primary signal for factual accuracy. A user with low bot scores can still share misinformation, and a high-follower account does not guarantee truthfulness.]
- **What the confusion matrix reveals**: [Discuss false positives — True claims predicted False — vs. false negatives — False claims predicted True. In a misinformation detection context, false negatives (missing misinformation) are typically more costly.]
- **Potential improvements**: Increasing TF-IDF vocabulary, tuning dropout rates, adding early stopping, or combining both feature sets into a single model.